# 🤖 1. Naive RAG에서 Agentic RAG로: 검색을 '단계'에서 '도구'로

이 노트북은 고정된 RAG 파이프라인과 에이전트 기반 RAG의 차이를 **같은 데이터, 같은 질문**으로 비교합니다.
마지막에는 `app/agents/tool_rag_agent/`가 이 노트북에서 만든 에이전트와 똑같은 구조라는 것을 확인하고, Mission 01로 넘어갑니다.

---

### 🎯 학습 목표
1. Naive RAG(검색 → 생성 고정 파이프라인)를 직접 만들고, **복합 질문에서 실패하는 지점**을 확인합니다.
2. Agentic RAG를 **"검색 과정의 판단을 LLM에게 맡긴 구조"** 로 설명할 수 있습니다.
3. 사전 구축된 4대 검색 도구를 `create_agent`에 연결해 **도구를 선택하고 여러 번 호출하는** ReAct 에이전트를 만듭니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 루트 경로, `.env`, `nest_asyncio`, LLM 초기화 |
| **Part 1** | Naive RAG 파이프라인 | Retriever + Prompt + LLM 체인, 단순 질문 성공 |
| **Part 2** | Naive RAG의 한계 | 다른 데이터 소스, 여러 단계 추론이 필요한 질문에서 실패 |
| **Part 3** | Agentic RAG의 핵심 아이디어 | 검색 = 도구, 판단 지점 5가지 |
| **Part 4** | 준비된 검색 도구 살펴보기 | 사내 규정, 한국은행 보고서, 조직 그래프, GraphRAG |
| **Part 5** | ReAct 에이전트 구축 및 테스트 | 도구 선택, 다단계 호출, 검색 생략, 멀티턴 |
| **Part 6** | 정리와 Mission 01 | `app/agents/tool_rag_agent/` 구조와 미션 안내 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)

print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 공통 유틸리티 로드
from app.utils import init_chat_model, normalize_content

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)
print(f"✅ LLM 초기화 완료: {llm.model}")

---
## 📚 Part 1. Naive RAG 파이프라인

Naive RAG는 **검색 → 프롬프트 결합 → 생성**의 순서가 코드로 고정된 파이프라인입니다.

```
사용자 질문 ──▶ Retriever (top-k) ──▶ Prompt { context + question } ──▶ LLM ──▶ 답변
```

데이터는 이미 `app/database/chroma_db`에 인덱싱되어 있습니다. (인덱싱 과정은 2교시 `2_Indexing.ipynb`에서 다룹니다.)
여기서는 **사내 규정 컬렉션** 하나를 검색기로 불러와 체인을 구성합니다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
policy_store = Chroma(
    collection_name="enterprise_policy_store",
    embedding_function=embeddings,
    persist_directory="app/database/chroma_db",
)
policy_retriever = policy_store.as_retriever(search_kwargs={"k": 2})

docs = policy_retriever.invoke("당일 출장 일비 기준")
for d in docs:
    print("📄", d.metadata.get("breadcrumb"))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_template("""당신은 (주)넥스트AI 사내 규정 Q&A 어시스턴트입니다.
아래 [문서]에 근거해서 답하세요.

[문서]
{context}

[질문]
{question}""")


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


def naive_rag(question: str) -> str:
    """검색 → 프롬프트 → 생성이 고정된 Naive RAG 파이프라인"""
    context = format_docs(policy_retriever.invoke(question))
    return (rag_prompt | llm | StrOutputParser()).invoke({"context": context, "question": question})


print(naive_rag("팀원이 당일 출장을 가면 일비는 얼마를 받나요?"))

✅ 질문이 **한 문서의 한 조항**으로 답할 수 있는 경우 Naive RAG는 잘 동작합니다.

---
## 🧱 Part 2. Naive RAG의 한계

이번에는 실제 업무에서 자주 나오는 **복합 질문**을 던져 보겠습니다.

> *"클라우드운영팀 김철수 수석의 본부장은 누구이고, 그 본부장의 결재가 필요한 프로젝트 예산 기준은 얼마인가요?"*

이 질문에 답하려면 다음 순서가 필요합니다.
1. **조직 데이터**에서 김철수 수석 → 클라우드운영팀 → 클라우드사업본부 → 본부장을 따라가고
2. **결재 규정**에서 본부장 승인이 필요한 예산 구간을 찾아야 합니다.

In [ ]:
complex_question = "클라우드운영팀 김철수 수석의 본부장은 누구이고, 그 본부장의 결재가 필요한 프로젝트 예산 기준은 얼마인가요?"

print("🔍 Naive RAG가 가져온 문서:")
for d in policy_retriever.invoke(complex_question):
    print("  📄", d.metadata.get("breadcrumb"))

print("\n💬 Naive RAG 답변:")
print(naive_rag(complex_question))

### 💡 왜 실패할까요?

| 한계 | 설명 |
|:---|:---|
| **단일 소스** | 파이프라인에 연결된 검색기는 사내 규정 하나뿐입니다. 조직도는 처음부터 볼 수 없습니다. |
| **단발 검색** | 질문 전체로 한 번만 검색합니다. "본부장 찾기 → 예산 기준 찾기"처럼 **앞 결과를 보고 다음 검색어를 정하는** 일이 불가능합니다. |
| **판단 부재** | 검색 결과가 질문에 충분한지 판단하지 않고 그대로 생성 단계로 넘깁니다. |

검색기를 여러 개 붙이고 분기 로직을 코드로 짜면 일부 해결되지만, **질문 유형이 늘어날 때마다 분기 코드가 늘어납니다.**

---
## 🧭 Part 3. Agentic RAG의 핵심 아이디어

Agentic RAG는 **검색을 파이프라인의 한 단계가 아니라, 에이전트가 필요할 때 호출하는 도구로** 바꿉니다.
그러면 검색 과정의 판단을 LLM이 맡게 됩니다.

```
             ┌──────────────── ReAct Loop ────────────────┐
사용자 질문 ─▶│ 생각(Thought) ─▶ 도구 호출(Action) ─▶ 관찰(Observation) │─▶ 답변
             └──────────────▲──────────────────────────┘
                            └── 결과가 부족하면 다시 검색
```

| 판단 지점 | 에이전트가 스스로 결정하는 것 |
|:---|:---|
| **검색 여부** | 이 질문에 검색이 필요한가? (인사말에는 검색하지 않음) |
| **소스 선택** | 사내 규정, 산업 보고서, 조직 그래프 중 어디를 볼까? |
| **질의 변형** | 질문을 어떤 검색어로, 몇 개로 나눠서 검색할까? |
| **결과 평가** | 가져온 결과로 답하기에 충분한가? |
| **종료** | 언제 검색을 멈추고 답할까? |

> [!IMPORTANT]
> 에이전트 루프 자체는 `create_agent` 한 줄로 만들어집니다.
> **Agentic RAG의 설계 품질은 루프가 아니라, 에이전트가 쓰는 도구와 그 뒤의 데이터 계층에서 결정됩니다.**
> 2교시(인덱싱), 3교시(MCP & Skills), 4교시(평가)가 모두 이 부분을 다룹니다.

---
## 🔧 Part 4. 준비된 검색 도구 살펴보기

`app/agents/tool_rag_agent/tools.py`에는 4대 데이터베이스를 감싼 검색 도구가 준비되어 있습니다.
LLM이 보는 것은 함수 코드가 아니라 **도구 이름, 설명(Docstring), 인자 스키마**입니다.

In [ ]:
from app.agents.tool_rag_agent.tools import (
    search_company_policy,
    search_bok_reports,
    search_graph_relations,
    query_enterprise_graphrag,
)

retrieval_tools = [search_company_policy, search_bok_reports, search_graph_relations, query_enterprise_graphrag]

for t in retrieval_tools:
    print(f"🔧 {t.name}")
    print(f"   설명: {t.description.splitlines()[0]}")
    print(f"   인자: {list(t.args.keys())}\n")

In [ ]:
# 도구는 일반 함수처럼 직접 호출해 볼 수 있습니다.
print(search_graph_relations.invoke({"seed_entity": "김철수 수석", "max_hops": 2}))

In [ ]:
print(search_bok_reports.invoke({"query": "2024년 반도체 수출 실적"})[:700])

---
## 🤖 Part 5. ReAct 에이전트 구축 및 테스트

`create_agent`에 **모델, 도구, 시스템 프롬프트, 체크포인터**만 연결하면 에이전트가 완성됩니다.
도구별 호출 지침은 시스템 프롬프트가 아니라 **각 도구의 설명**에 들어 있다는 점에 주목하세요.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

SYSTEM_PROMPT = """당신은 (주)넥스트AI의 엔터프라이즈 멀티소스 Q&A 에이전트입니다.
각 도구의 설명에 적힌 지침을 따라 자율적으로 검색하세요.
도구 결과에 근거해서만 답하고, 결과에 없는 내용은 "확인되지 않는다"고 답하세요.
답변에는 규정 조항 번호, 수치, 출처를 명시하세요."""

agent = create_agent(
    model=llm,
    tools=retrieval_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)


def ask(question: str, thread_id: str = "demo"):
    """에이전트를 실행하고 도구 호출 궤적(Trajectory)과 최종 답변을 출력합니다."""
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        {"configurable": {"thread_id": thread_id}},
    )
    print(f"❓ {question}\n")
    # 체크포인터가 이전 대화까지 돌려주므로, 이번 질문 이후의 메시지만 봅니다.
    last_human = max(i for i, m in enumerate(result["messages"]) if m.type == "human")
    turn = result["messages"][last_human:]
    calls = [tc for m in turn for tc in (getattr(m, "tool_calls", None) or [])]
    for i, tc in enumerate(calls, 1):
        print(f"  🔄 Step {i}: {tc['name']}({tc['args']})")
    print(f"  ➜ 도구 호출 {len(calls)}회\n")
    print("💬", normalize_content(result["messages"][-1].content))
    return result

print("✅ ReAct 에이전트 생성 완료")

### 5.1 Naive RAG가 실패했던 복합 질문

에이전트가 **조직 그래프 → 사내 규정** 순서로 도구를 고르고, 앞 결과를 보고 다음 검색어를 정하는지 확인하세요.

In [ ]:
_ = ask(complex_question, thread_id="t-complex")

### 5.2 같은 소스를 여러 번 나눠 검색하기

분기 비교 질문입니다. `search_bok_reports`의 설명에 적힌 "한 번에 1개 분기만" 지침에 따라 **분기별로 나눠 호출**하는지 봅니다.

In [ ]:
_ = ask("2024년 1분기와 3분기의 반도체 수출 동향을 비교해줘.", thread_id="t-bok")

### 5.3 검색이 필요 없는 질문

Naive RAG는 인사말에도 무조건 검색합니다. 에이전트는 **검색 여부**부터 판단합니다.

In [ ]:
_ = ask("안녕하세요! 어떤 질문을 할 수 있나요?", thread_id="t-hello")

### 5.4 멀티턴 대화

체크포인터(`InMemorySaver`)와 같은 `thread_id`를 쓰면 이전 대화를 기억합니다.
앞 질문(5.1)의 맥락에서 이어지는 질문을 던져 봅니다.

In [ ]:
_ = ask("그 본부장이 승인권자인 프로젝트는 무엇이고 예산은 얼마인가요?", thread_id="t-complex")

---
## 🏁 Part 6. 정리와 Mission 01

| 구분 | Naive RAG | Agentic RAG (ReAct + 검색 도구) |
|:---|:---|:---|
| 제어 흐름 | 코드에 고정 (검색 → 생성) | LLM이 루프 안에서 판단 |
| 데이터 소스 | 파이프라인에 연결된 1개 | 도구로 등록된 여러 개 중 선택 |
| 검색 횟수 | 1회 고정 | 0회 ~ N회 |
| 설계 포인트 | 프롬프트, 검색기 튜닝 | **도구 설계 + 데이터 계층** |

### 📦 이 에이전트는 app에 이미 있습니다

방금 만든 에이전트는 `app/agents/tool_rag_agent/`와 같은 구조입니다. 서비스 코드는 세 파일로 나뉩니다.

```
app/agents/tool_rag_agent/
 ├── tools.py   ← Part 4에서 본 4대 검색 도구
 ├── prompt.py  ← 시스템 프롬프트
 └── agent.py   ← create_agent 조립 (모델 + 도구 + 프롬프트 + SQLite 체크포인터)
```

In [ ]:
with open("app/agents/tool_rag_agent/agent.py", "r", encoding="utf-8") as f:
    print(f.read())

### 🎯 Mission 01. Tool RAG Agent 서비스 실행 및 검증

`missions/01_missions.md`의 미션을 수행하세요.
1. FastAPI 서버와 Chainlit UI를 띄우고 `chatbot`으로 통신을 확인합니다.
2. `app/agents/tool_rag_agent/agent.py`의 에이전트 조립 코드를 완성합니다.
3. 웹 UI에서 이 노트북의 질문들을 다시 던지고, 도구 호출 단계가 화면에 어떻게 보이는지 확인합니다.